In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit provide or receive gifts or anything of 
   value to/from OR pay for the travel, hospitality or entertainment of Customers 
   and Third Parties?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. TARGET CATEGORY CODES: Grabs the full list of Category Codes from the ABAC 
      category_codes_database. Any code in this table is considered flagged.
   3. COUPA DATA (7 Files): Unions the 7 Coupa files. Parses the 'Account' string 
      (format: xxxx-xxxx-xxxxxx) to extract the Cost Center (first block) and Category 
      Code (last block). Uses TRY_CAST to INT to safely strip leading zeros.
   4. FINANCE DATA (4 Files verified): Unions the Finance files. Extracts the Cost Center 
      (casting to INT) and Category Code directly from their respective columns.
   5. CONSOLIDATION & FILTERING: Combines Coupa and Finance data into one master list 
      of transactions. INNER JOINS to the ABAC category_codes_database to strictly keep 
      transactions that match a flagged Category Code.
   6. MAPPING & OUTPUT: Maps flagged transactions to AU_IDs using the CC Bootstrap. 
      LEFT JOINS this mapped data back to the Master AU anchor. If an AU has any mapped 
      transactions, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List -- All rows here are in the list by definition
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Target_Category_Codes AS (
    -- 2. Grab the ABAC category codes (Any transaction matching these is a "Yes")
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3a. Union all 7 Coupa files into one master dataset
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3b. Parse the Coupa Account string: splits by '-' 
    -- [0] is the first part (CC), [2] is the third part (Category Code)
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%' -- Ensures the string actually has the correct hyphen format
),

Combined_Finance_Raw AS (
    -- 4a. Union all available Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    -- Uncomment below if files 5 and 6 exist!
    -- UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    -- UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 4b. Clean the Finance columns
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 5a. Stack Coupa and Finance data together
    SELECT Cleaned_CC, CatCode, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Cleaned_CC, CatCode, Source_System FROM Finance_Parsed
),

Flagged_Transactions AS (
    -- 5b. Strictly keep transactions where the Category Code matches the ABAC database
    SELECT 
        t.Cleaned_CC,
        t.CatCode,
        t.Source_System
    FROM All_Transactions t
    INNER JOIN Target_Category_Codes c
        ON t.CatCode = c.CatCode
    WHERE t.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    -- 6a. Standardize the CC Mapping table
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 6b. Map the flagged transactions to their respective AU_IDs
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 7. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA01' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 Drill-Down
   
   PURPOSE: Shows EVERY flagged transaction from Coupa/Finance, regardless of whether 
   the Cost Center maps to an AU, or whether that AU exists in the Master List.
   Useful for finding unmapped Cost Centers generating high-risk transactions.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Target_Category_Codes AS (
    SELECT DISTINCT TRIM(CatCode) AS CatCode
    FROM hive_metastore.ra_adido_2025.category_codes_database
    WHERE CatCode IS NOT NULL
),

Coupa_Parsed AS (
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRIM(SPLIT(Account, '-')[2]) AS CatCode,
        'Coupa' AS Source_System
    FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered WHERE Account LIKE '%-%-%'
    UNION ALL SELECT Account, TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT), TRIM(SPLIT(Account, '-')[2]), 'Coupa' FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered WHERE Account LIKE '%-%-%'
),

Finance_Parsed AS (
    SELECT 
        CostCenter AS Raw_String,
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System
    FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, TRY_CAST(TRIM(CostCenter) AS INT), TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, TRY_CAST(TRIM(CostCenter) AS INT), TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, TRY_CAST(TRIM(CostCenter) AS INT), TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_4
),

All_Transactions AS (
    SELECT Raw_String, Cleaned_CC, CatCode, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode, Source_System FROM Finance_Parsed
),

Flagged_Transactions AS (
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.CatCode,
        t.Source_System
    FROM All_Transactions t
    INNER JOIN Target_Category_Codes c
        ON t.CatCode = c.CatCode
    WHERE t.Cleaned_CC IS NOT NULL
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    f.Cleaned_CC AS Flagged_Cost_Center,
    f.CatCode AS Flagged_Category_Code,
    f.Source_System,
    f.Raw_String AS Original_Data_Value
FROM Flagged_Transactions f
-- LEFT JOIN to mapping so we see unmapped CCs
LEFT JOIN CC_Mapping m
    ON f.Cleaned_CC = m.Mapped_CC
-- LEFT JOIN to Master List so we know if the AU is actually in scope
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
ORDER BY BusinessID, f.Cleaned_CC;